# QIGOA Reality-Check — Kaggle runner

Notebook chạy pipeline QIGOA Reality-Check trên Kaggle. **Đọc `docs/huong-dan-setup-kaggle.md` trước.**

**Quy tắc bất khả xâm phạm (CLAUDE.md §2):**
- Số synthetic (smoke test) **không bao giờ** là kết quả — luôn `[PLACEHOLDER]`.
- Mọi con số vào paper phải truy được về `results/` qua `docs/RESULTS.md`.
- **CẤM chạy E2 (lưới chính) trước** khi cổng bit-exact (E1) PASS và 4 thí nghiệm rủi ro Tuần-1 còn sống (preregistration §6/A10).
- Notebook này **KHÔNG nhúng số kết quả** — nó chỉ chạy script và ghi `results/`. Output cell không phải nguồn số liệu; `results/*.csv` mới là.

**Thứ tự cell:** (a) setup + kiểm tra `/kaggle/input` → (b) **cờ chọn experiment** → (c) **3 cổng cứng pytest — FAIL thì DỪNG** → (d) smoke test synthetic → (e) E1 cổng bit-exact → (f) E2 theo stage `--k-subset` (resume-được) → (g) zip `results/` để tải về → (h) in `run-manifest.json`.

## (a) Setup — clone code + cài package thiếu + kiểm tra dataset đã mount

Cần **Internet: ON** (Settings phải). GPU để **None** cho mọi thứ trừ `run_unet.py` (tiết kiệm 30 GPU-h/tuần).

Thay `<TEN_REPO>` bằng tên repo thật. Ghi lại commit hash — nó vào `run-manifest.json`.

In [ ]:
# Clone repo (Cách A). Nếu dùng utility-dataset (Cách B) thì thay bằng cp -r từ /kaggle/input/.
!git clone https://github.com/HoangLong08/<TEN_REPO>.git /kaggle/working/qigoa
%cd /kaggle/working/qigoa
COMMIT = !git rev-parse HEAD
print('git commit:', COMMIT[0] if COMMIT else 'UNKNOWN (not a git checkout)')

# Kaggle đã có sẵn numpy/scipy/scikit-image/scikit-learn/pandas/nibabel/PyYAML/matplotlib/torch/tqdm.
# Thường chỉ cần cài thêm medpy (HD95/NSD) + tifffile (LGG 3 kênh):
!pip install -q medpy tifffile
# Nếu medpy lỗi build: !pip install -q git+https://github.com/deepmind/surface-distance.git

In [ ]:
# Kiem tra /kaggle/input da mount + liet ke dataset (Buoc 1). /kaggle/input/ la READ-ONLY.
import os

INPUT_ROOT = '/kaggle/input'
EXPECTED = {
    'brats20-dataset-training-validation': 'CHINH (E1-E6)',
    'lgg-mri-segmentation': 'E7 - xem canh bao A8 truoc khi goi la external validation',
}

if not os.path.isdir(INPUT_ROOT):
    print('KHONG THAY /kaggle/input -> notebook nay dang chay ngoai Kaggle?')
else:
    print('Dataset dang mount tai /kaggle/input:')
    for d in sorted(os.listdir(INPUT_ROOT)):
        print('   -', d)
    print()
    for slug, role in EXPECTED.items():
        ok = os.path.isdir(os.path.join(INPUT_ROOT, slug))
        print(('OK   ' if ok else 'THIEU'), slug, '|', role)

print('\nGhi ket qua vao /kaggle/working/ (mat khi het session -> phai tai ve, cell (g)).')

## (b) Cờ chọn experiment

Đặt `EXPERIMENT` rồi chạy các cell dưới. Thứ tự hợp lệ **bị ép** theo preregistration §6/A10 — cell (c) và (e) là cổng cứng, không bỏ qua được.

| `EXPERIMENT` | Script | Compute | Điều kiện tiên quyết |
|---|---|---|---|
| `smoke` | `run_exact_check.py --config configs/exp_smoke.yaml` | < 60s CPU | — (số là `[PLACEHOLDER]`) |
| `e1_exact` | `run_exact_check.py --config configs/exp_main.yaml` | < 30ph CPU | pytest gate PASS |
| `week1_risky` | 4 thí nghiệm rủi ro (A10) | ~3h CPU | E1 PASS |
| `e2_main` | `run_main_grid.py --k-subset ...` | ~20h CPU, chia stage | E1 PASS **và** cả 4 rủi ro sống |
| `e3_ablation` | `run_ablation.py` | ~6h CPU | E2 xong |
| `e4_ceiling` | `run_ceiling.py` | ~2h CPU | E2 xong |
| `e4_unet` | `run_unet.py` | ~2h **GPU** | E4 ceiling xong |
| `e6_stats` | `run_stats.py` | phút | E2/E3/E4 xong |
| `e7_external` | `run_external.py` | ~8h CPU | ⚠️ đọc A8 trước |

In [ ]:
# ===== CO CHON EXPERIMENT =====
EXPERIMENT = 'smoke'   # smoke | e1_exact | week1_risky | e2_main | e3_ablation | e4_ceiling | e4_unet | e6_stats | e7_external
K_SUBSET   = '2,3,4'   # chi dung cho e2_main: chia stage theo k (session <= 12h)

CONFIG = {
    'smoke':        'configs/exp_smoke.yaml',
    'e1_exact':     'configs/exp_main.yaml',
    'week1_risky':  'configs/exp_ceiling.yaml',
    'e2_main':      'configs/exp_main.yaml',
    'e3_ablation':  'configs/exp_ablation.yaml',
    'e4_ceiling':   'configs/exp_ceiling.yaml',
    'e4_unet':      'configs/exp_unet.yaml',
    'e6_stats':     'configs/exp_main.yaml',
    'e7_external':  'configs/exp_external.yaml',
}[EXPERIMENT]

print('EXPERIMENT =', EXPERIMENT, '| config =', CONFIG)
if EXPERIMENT == 'e4_unet':
    print('>> Bat Accelerator: GPU T4 cho experiment nay. Moi experiment khac de None.')
elif EXPERIMENT == 'e2_main':
    print('>> CHI chay khi E1 PASS va ca 4 thi nghiem rui ro Tuan-1 con song (A10). K_SUBSET =', K_SUBSET)

## (c) 3 cổng cứng pytest — chạy TRƯỚC mọi thứ, FAIL thì DỪNG

`test_exact_dp` (DP khớp vét cạn) · `test_nfe_budget` (mọi thuật toán tiêu đúng cùng NFE, ±0) · `test_degeneracy` (unit test số học của cơ chế suy biến — **không phải** một Result).

> 🔴 Cell này **raise SystemExit** nếu bất kỳ test nào fail. Không chạy tiếp. `test_nfe_budget` fail ⇒ **lưới vô hiệu** (đó chính là lỗi đã giết lô cũ: thừa 13,4% NFE).

In [ ]:
import subprocess, sys

GATES = ['tests/test_exact_dp.py', 'tests/test_nfe_budget.py', 'tests/test_degeneracy.py']
proc = subprocess.run([sys.executable, '-m', 'pytest', *GATES, '-q'], text=True)

if proc.returncode != 0:
    raise SystemExit(
        'CONG CUNG FAIL -> DUNG TOAN BO. Khong chay bat ky experiment nao.\n'
        ' - test_exact_dp fail  : moi ket luan ve P2 dung tren DP. Debug DAU TIEN = audit quy uoc\n'
        '                         (0log0, lop rong, co tinh nen cuong-do-0 khong) — KHONG phai debug DP (A5b/A5c).\n'
        ' - test_nfe_budget fail: LUOI VO HIEU (day la loi da giet lo cu: +13,4% NFE).\n'
        ' - test_degeneracy fail: co che suy bien khong dung nhu phat bieu -> doc lai prereg §6/A1.'
    )
print('\n3 CONG CUNG PASS -> duoc phep chay experiment.')

## (d) Smoke test — synthetic, < 60s

Chạy pipeline trên dữ liệu synthetic để bắt lỗi **cấu trúc** trước khi đốt giờ Kaggle trên dữ liệu thật.

> ⚠️ **Mọi số ở cell này là `[PLACEHOLDER]`** — synthetic phantom, **KHÔNG phải kết quả**, không bao giờ vào paper, không bao giờ vào `docs/RESULTS.md` như một run thật.

In [ ]:
!python scripts/run_exact_check.py --config configs/exp_smoke.yaml
print('\n[PLACEHOLDER] So o tren la synthetic — chi xac nhan pipeline chay end-to-end.')

## (e) E1 — cổng bit-exact ★ CỔNG CỨNG THỨ HAI (trên dữ liệu THẬT)

DP phải khớp vét cạn tại k=2,3 theo quy ước A5a: `|f_DP − f_brute| ≤ 1e-9` **và** mask cảm sinh giống hệt (ngưỡng canonicalise).

> 🔴 **FAIL → DỪNG TOÀN BỘ.** Cho ra **Bảng II** (`results/exact/dp_vs_bruteforce.csv`).

In [ ]:
import os, subprocess, sys

proc = subprocess.run([sys.executable, 'scripts/run_exact_check.py', '--config', 'configs/exp_main.yaml'], text=True)
csv = 'results/exact/dp_vs_bruteforce.csv'

if proc.returncode != 0 or not os.path.exists(csv):
    raise SystemExit('E1 FAIL (hoac khong sinh ' + csv + ') -> DUNG TOAN BO. Audit quy uoc histogram truoc (A5c).')
print('\nE1 PASS. Da sinh:', csv)
print('=> Them dong provenance vao docs/RESULTS.md, roi moi sang E1b / Tuan-1 (4 thi nghiem rui ro).')

## (f) E2 — lưới chính theo `--k-subset`, resume-được

> 🔴 **CHỈ chạy khi:** E1 PASS **và** cả 4 thí nghiệm rủi ro Tuần-1 còn sống (Bảng I κ · Bậc-5 nested CV · Spearman(k,Dice) theo từng rule · morph vs oracle). Xem `docs/huong-dan-setup-kaggle.md` Bước 3.3.

Session ≤ 12h ⇒ E2 (~20h) chia theo `k`. Script checkpoint sau mỗi `(image, k, algo, seed)` vào `results/main/partial/` → session chết chỉ chạy lại phần chưa xong. GPU = None.

In [ ]:
# Chay mot stage theo K_SUBSET da dat o cell (b). Vi du: session 1 -> '2,3,4'; session 2 -> '5,6,8,10'.
assert EXPERIMENT == 'e2_main', 'Dat EXPERIMENT = "e2_main" o cell (b) truoc khi chay cell nay.'
!python scripts/run_main_grid.py --config {CONFIG} --k-subset {K_SUBSET}

# Da chay het cac k? -> merge partial + thong ke + build bang/hinh tu results/ (khong go so tay).
# !python scripts/run_stats.py --config configs/exp_main.yaml
# !python scripts/build_tables.py
# !python scripts/build_figures.py

## (g) Đóng gói `results/` để tải về

`/kaggle/working/` **mất khi session hết hạn**. Đóng gói `results/` thành `.tgz`, tải từ tab **Output**, giải nén vào repo local, **thêm dòng provenance vào `docs/RESULTS.md`**, commit. Không xoá run xấu — run âm là dữ liệu.

In [ ]:
import time
stamp = time.strftime('%Y%m%d_%H%M')
!cd /kaggle/working/qigoa && tar czf /kaggle/working/results_{stamp}.tgz results/
print('Da dong goi: /kaggle/working/results_' + stamp + '.tgz')
print('Tai ve tu tab Output -> giai nen vao repo -> them provenance vao docs/RESULTS.md -> commit.')

## (h) In `run-manifest.json`

**Không có manifest = run không tồn tại với paper** (CLAUDE.md §5.2). Manifest do `src/manifest.py` sinh cho MỖI session: `{git_commit, config_hash, seeds, dataset_version, lib_versions, timestamp, output_paths}`. Copy `git_commit` + `seeds` vào dòng provenance trong `docs/RESULTS.md`.

In [ ]:
import glob, json

manifests = sorted(glob.glob('results/**/run-manifest.json', recursive=True))
if not manifests:
    print('KHONG TIM THAY run-manifest.json trong results/.')
    print('=> Run nay KHONG TON TAI voi paper (CLAUDE.md §5.2). Kiem tra script co goi src/manifest.py khong.')
else:
    for m in manifests:
        print('=' * 70)
        print(m)
        print('=' * 70)
        print(json.dumps(json.load(open(m)), indent=2, ensure_ascii=False))
    print('\nDan dong provenance vao docs/RESULTS.md theo mau:')
    print('  Bang X <- results/<path>.csv <- scripts/<script>.py --config <config> @commit <hash>, seeds {..}')